In [ ]:
import csv
import json
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from torchvision.models import inception_v3, Inception_V3_Weights

from datasets import load_dataset
from tqdm.auto import tqdm

from ignite.metrics import FID, InceptionScore
from pytorch_fid.inception import InceptionV3

In [ ]:
PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "dcgan_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 64
NZ = 100
NGF = 64
NDF = 64

BATCH_SIZE = 64
EPOCHS = 100
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999

MAX_IMAGES = None
TEST_FRACTION = 0.1
EVAL_IMAGES = 5000
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0

RESUME = False

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Output directory:", OUTPUT_DIR)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Currently training on CPU.")

In [ ]:
from datasets import Dataset as HFDataset, concatenate_datasets

DATA_DIR = PROJECT_DIR / "dataset"
DATA_DIR.mkdir(exist_ok=True)
arrow_files = sorted(DATA_DIR.glob("wikiart-train-*.arrow"))

if arrow_files:
    raw_dataset = concatenate_datasets([
        HFDataset.from_file(str(path)) for path in arrow_files
    ])
else:
    raw_dataset = load_dataset(
        "huggan/wikiart",
        revision="d559852d2b232e0fcf195e775866964f0564f2b5",
        split="train",
        cache_dir=str(DATA_DIR / "hf_cache"),
    )

if len(raw_dataset) != 81_444:
    raise ValueError(f"Expected 81,444 WikiArt images, got {len(raw_dataset):,}")
print("Total images:", len(raw_dataset))


In [ ]:
resize_crop = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
])

cache_path = DATA_DIR / (
    f"wikiart_{IMAGE_SIZE}x{IMAGE_SIZE}_{raw_dataset._fingerprint}.npy"
)
expected_shape = (len(raw_dataset), IMAGE_SIZE, IMAGE_SIZE, 3)

if not cache_path.exists():
    temporary_path = cache_path.with_suffix(".tmp.npy")
    output = np.lib.format.open_memmap(
        temporary_path, mode="w+", dtype=np.uint8, shape=expected_shape
    )
    try:
        for index, row in enumerate(tqdm(
            raw_dataset, total=len(raw_dataset), desc="Preprocessing WikiArt"
        )):
            image = row["image"].convert("RGB")
            output[index] = np.asarray(resize_crop(image), dtype=np.uint8)
        output.flush()
    finally:
        del output

    temporary_path.replace(cache_path)

cached_images = np.load(cache_path, mmap_mode="r")
if cached_images.shape != expected_shape or cached_images.dtype != np.uint8:
    raise ValueError(f"Unexpected cache shape or dtype: {cache_path}")

print("Cache:", cache_path)
print("Shape:", cached_images.shape, "| dtype:", cached_images.dtype)


In [ ]:
row_ids = HFDataset.from_dict({"row_id": np.arange(len(raw_dataset))})
split_ids = row_ids.train_test_split(test_size=TEST_FRACTION, seed=SEED)


class CachedWikiArtDataset(Dataset):
    def __init__(self, images, indices):
        self.images = images
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image = self.images[self.indices[index]].copy()
        tensor = torch.from_numpy(image).permute(2, 0, 1)
        return tensor.float().div_(255.0).sub_(0.5).div_(0.5)


train_dataset = CachedWikiArtDataset(cached_images, split_ids["train"]["row_id"])
test_dataset = CachedWikiArtDataset(cached_images, split_ids["test"]["row_id"])

train_batch_size = min(BATCH_SIZE, len(train_dataset))
train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Batches per epoch:", len(train_loader))


In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64):
        super().__init__()

        self.main = nn.Sequential(
            # 1×1 → 4×4
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # 4×4 → 8×8
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # 8×8 → 16×16
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # 16×16 → 32×32
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # 32×32 → 64×64
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.main(z)


class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super().__init__()

        self.main = nn.Sequential(
            # 64×64 → 32×32
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32×32 → 16×16
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16×16 → 8×8
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8×8 → 4×4
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 4×4 → 1×1
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)
        )

    def forward(self, image):
        return self.main(image).flatten()


def initialize_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, mean=1.0, std=0.02)
        nn.init.zeros_(module.bias)


netG = Generator(NZ, NGF).to(device)
netD = Discriminator(NDF).to(device)

netG.apply(initialize_weights)
netD.apply(initialize_weights)

criterion = nn.BCEWithLogitsLoss()

optimizerG = torch.optim.Adam(
    netG.parameters(), lr=LR, betas=(BETA1, BETA2)
)

optimizerD = torch.optim.Adam(
    netD.parameters(), lr=LR, betas=(BETA1, BETA2)
)

fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

print("Generator parameters:",
      f"{sum(p.numel() for p in netG.parameters()):,}")
print("Discriminator parameters:",
      f"{sum(p.numel() for p in netD.parameters()):,}")

In [ ]:
config = {
    "dataset": "huggan/wikiart",
    "dataset_fingerprint": getattr(raw_dataset, "_fingerprint", None),
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "noise_dimension": NZ,
    "ngf": NGF,
    "ndf": NDF,
    "batch_size": train_batch_size,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "betas": [BETA1, BETA2],
    "max_images": MAX_IMAGES,
    "test_fraction": TEST_FRACTION,
    "train_images": len(train_dataset),
    "test_images": len(test_dataset),
    "preprocessing": "RGB, Resize(64), CenterCrop(64), normalize to [-1,1]",
    "torch_version": str(torch.__version__),
    "device": str(device)
}

checkpoint_path = OUTPUT_DIR / "checkpoint_latest.pt"
history = []
start_epoch = 0

if RESUME:
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    old_config = checkpoint["config"]
    for key in [
        "dataset", "dataset_fingerprint", "seed", "image_size",
        "noise_dimension", "ngf", "ndf", "batch_size",
        "learning_rate", "betas", "max_images",
        "test_fraction", "train_images", "test_images"
    ]:
        if old_config.get(key) != config.get(key):
            raise ValueError(f"Different configuration: {key}")

    netG.load_state_dict(checkpoint["generator"])
    netD.load_state_dict(checkpoint["discriminator"])
    optimizerG.load_state_dict(checkpoint["optimizerG"])
    optimizerD.load_state_dict(checkpoint["optimizerD"])

    fixed_noise = checkpoint["fixed_noise"].to(device)
    history = checkpoint["history"]
    start_epoch = checkpoint["epoch"]

    torch.set_rng_state(checkpoint["torch_rng"].cpu())
    np.random.set_state(checkpoint["numpy_rng"])
    random.setstate(checkpoint["python_rng"])

    if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(
            [state.cpu() for state in checkpoint["cuda_rng"]]
        )

    print(f"Resume from completed epoch {start_epoch}")

with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()

    netG.train()
    netD.train()

    total_g_loss = 0.0
    total_d_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for real_images in progress:
        real_images = real_images.to(device, non_blocking=True)
        batch_size = real_images.size(0)

        real_targets = torch.ones(batch_size, device=device)
        fake_targets = torch.zeros(batch_size, device=device)

        optimizerD.zero_grad(set_to_none=True)

        real_logits = netD(real_images)
        d_loss_real = criterion(real_logits, real_targets)

        noise = torch.randn(batch_size, NZ, 1, 1, device=device)
        fake_images = netG(noise)

        fake_logits = netD(fake_images.detach())
        d_loss_fake = criterion(fake_logits, fake_targets)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizerD.step()

        for parameter in netD.parameters():
            parameter.requires_grad_(False)

        optimizerG.zero_grad(set_to_none=True)

        generated_logits = netD(fake_images)
        g_loss = criterion(generated_logits, real_targets)

        g_loss.backward()
        optimizerG.step()

        for parameter in netD.parameters():
            parameter.requires_grad_(True)

        total_g_loss += g_loss.item() * batch_size
        total_d_loss += d_loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            G=f"{g_loss.item():.4f}",
            D=f"{d_loss.item():.4f}"
        )

    epoch_record = {
        "epoch": epoch + 1,
        "g_loss": total_g_loss / total_samples,
        "d_loss": total_d_loss / total_samples,
        "seconds": time.time() - epoch_start
    }
    history.append(epoch_record)

    netG.eval()
    with torch.no_grad():
        samples = netG(fixed_noise).cpu()

    save_image(
        samples,
        OUTPUT_DIR / f"epoch_{epoch + 1:03d}.png",
        nrow=8,
        normalize=True,
        value_range=(-1, 1)
    )

    with open(
        OUTPUT_DIR / "training_history.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    checkpoint = {
        "epoch": epoch + 1,
        "config": config,
        "generator": netG.state_dict(),
        "discriminator": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "fixed_noise": fixed_noise.cpu(),
        "history": history,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if device.type == "cuda" else None
        )
    }

    temporary_path = OUTPUT_DIR / "checkpoint_latest.tmp"
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(checkpoint_path)

    print(
        f"Epoch {epoch + 1}: "
        f"G={epoch_record['g_loss']:.4f}, "
        f"D={epoch_record['d_loss']:.4f}, "
        f"time={epoch_record['seconds']:.1f}s"
    )

torch.save(netG.state_dict(), OUTPUT_DIR / "generator_final.pt")

print("Training finished.")

In [ ]:
if not history:
    raise RuntimeError("No training before.")

epochs = [row["epoch"] for row in history]
g_losses = [row["g_loss"] for row in history]
d_losses = [row["d_loss"] for row in history]

plt.figure(figsize=(8, 4.5))
plt.plot(epochs, g_losses, label="Generator")
plt.plot(epochs, d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("DCGAN training losses")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "loss_curves.png", dpi=300)
plt.savefig(OUTPUT_DIR / "loss_curves.pdf")
plt.show()

In [ ]:
netG.eval()

sample_rng = torch.Generator(device=device).manual_seed(342)

with torch.no_grad():
    new_noise = torch.randn(
        64, NZ, 1, 1,
        device=device,
        generator=sample_rng
    )
    generated_images = netG(new_noise).cpu()

save_image(
    generated_images,
    OUTPUT_DIR / "generated_grid.png",
    nrow=8,
    normalize=True,
    value_range=(-1, 1)
)

individual_dir = OUTPUT_DIR / "generated_individual"
individual_dir.mkdir(exist_ok=True)

for index, image in enumerate(generated_images):
    save_image(
        image,
        individual_dir / f"painting_{index:03d}.png",
        normalize=True,
        value_range=(-1, 1)
    )

grid = make_grid(
    generated_images,
    nrow=8,
    normalize=True,
    value_range=(-1, 1)
)

plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis("off")
plt.title("Generated paintings")
plt.tight_layout()
plt.show()

In [ ]:
class FIDFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        block = InceptionV3.BLOCK_INDEX_BY_DIM[2048]

        self.model = InceptionV3(
            output_blocks=[block],
            resize_input=True,
            normalize_input=True
        )

    def forward(self, images):
        features = self.model(images)[0]
        return features.flatten(1)


class ISProbabilityExtractor(nn.Module):
    def __init__(self):
        super().__init__()

        weights = Inception_V3_Weights.DEFAULT
        self.preprocess = weights.transforms()
        self.model = inception_v3(weights=weights)

        self.model.transform_input = False

    def forward(self, images):
        images = self.preprocess(images)
        logits = self.model(images)
        return torch.softmax(logits, dim=1)


fid_extractor = FIDFeatureExtractor().to(device).eval()
is_extractor = ISProbabilityExtractor().to(device).eval()

for model in (fid_extractor, is_extractor):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

fid_metric = FID(
    num_features=2048,
    feature_extractor=fid_extractor,
    device=device
)

is_metric = InceptionScore(
    num_features=1000,
    feature_extractor=is_extractor,
    device=device
)

fid_metric.reset()
is_metric.reset()

n_eval = min(EVAL_IMAGES, len(test_dataset))

if n_eval < 2:
    raise ValueError("FID needs at least 2 images.")

if n_eval <= 2048:
    print(
        "Little images may lead to unrealiable results."
    )

eval_indices = np.random.default_rng(SEED + 1).permutation(
    len(test_dataset)
)[:n_eval]

eval_dataset = torch.utils.data.Subset(
    test_dataset,
    eval_indices.tolist()
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

eval_rng = torch.Generator(device=device).manual_seed(SEED + 1000)

netG.eval()

with torch.inference_mode():
    for real_images in tqdm(eval_loader, desc="Evaluating IS and FID"):
        real_images = real_images.to(device, non_blocking=True)
        current_batch_size = real_images.size(0)

        noise = torch.randn(
            current_batch_size, NZ, 1, 1,
            device=device,
            generator=eval_rng
        )

        fake_images = netG(noise)

        real_01 = ((real_images + 1) / 2).clamp(0, 1)
        fake_01 = ((fake_images + 1) / 2).clamp(0, 1)

        fid_metric.update((fake_01, real_01))
        is_metric.update(fake_01)

fid_score = float(fid_metric.compute())
inception_score = float(is_metric.compute())

metrics = {
    "FID": fid_score,
    "InceptionScore": inception_score,
    "num_real_images": n_eval,
    "num_generated_images": n_eval,
    "real_source": "held-out WikiArt test split",
    "fid_features": "pytorch-fid InceptionV3 pool3, 2048 dimensions",
    "is_model": "torchvision InceptionV3 ImageNet pretrained",
    "is_protocol": "single aggregate score, no split standard deviation",
    "completed_epochs": history[-1]["epoch"]
}

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Evaluated images: {n_eval}")
print(f"FID: {fid_score:.4f}")
print(f"Inception Score: {inception_score:.4f}")
print("Results saved to:", OUTPUT_DIR)